## 准备数据

In [37]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [38]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [39]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        ####################
        # 输入784维 (28*28)，隐藏层100维，输出10维
        # 使用He初始化（适合ReLU激活函数）
        self.W1 = tf.Variable(tf.random.normal([784, 100]) * tf.sqrt(2.0/784))
        self.b1 = tf.Variable(tf.zeros([100]))
        self.W2 = tf.Variable(tf.random.normal([100, 10]) * tf.sqrt(2.0/100))
        self.b2 = tf.Variable(tf.zeros([10]))
        
    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        ####################
        # 将28x28图片展平为784维向量
        x = tf.reshape(x, [-1, 784])
        # 第一层
        h1 = tf.matmul(x, self.W1) + self.b1
        h1 = tf.nn.relu(h1)
        # 第二层
        logits = tf.matmul(h1, self.W2) + self.b2
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [40]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.1*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [41]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 2.300394 ; accuracy 0.1558
epoch 1 : loss 2.1854944 ; accuracy 0.2243
epoch 2 : loss 2.0913324 ; accuracy 0.30578333
epoch 3 : loss 2.007957 ; accuracy 0.38381666
epoch 4 : loss 1.9316657 ; accuracy 0.44136667
epoch 5 : loss 1.8605171 ; accuracy 0.48428333
epoch 6 : loss 1.7934285 ; accuracy 0.51905
epoch 7 : loss 1.7297828 ; accuracy 0.54755
epoch 8 : loss 1.6689974 ; accuracy 0.57475
epoch 9 : loss 1.6108172 ; accuracy 0.59815
epoch 10 : loss 1.5551052 ; accuracy 0.61943334
epoch 11 : loss 1.5017753 ; accuracy 0.6389
epoch 12 : loss 1.4507511 ; accuracy 0.65671664
epoch 13 : loss 1.4021024 ; accuracy 0.67256665
epoch 14 : loss 1.3558143 ; accuracy 0.6868333
epoch 15 : loss 1.311906 ; accuracy 0.7005
epoch 16 : loss 1.2703168 ; accuracy 0.7123
epoch 17 : loss 1.2309963 ; accuracy 0.72353333
epoch 18 : loss 1.193874 ; accuracy 0.73338336
epoch 19 : loss 1.1588794 ; accuracy 0.7413333
epoch 20 : loss 1.1259407 ; accuracy 0.74908334
epoch 21 : loss 1.0949483 ; accuracy 0.7